# Atividade 01 - Exploracao dos dados

**Projeto:** priorizacao de leads de tecnologia na Bahia a partir dos dados abertos de CNPJ.

Este notebook carrega a amostra real da Receita Federal e produz a caracterizacao pedida no item 5 da Atividade 01. Ele nao reimplementa leitura nem regra: importa os mesmos modulos de `src/` que o pipeline da Atividade 02 usa, de modo que exploracao e producao nunca divirjam.

**Pre-requisito:** `python -m src.extract` (baixa a amostra; idempotente).

**Fonte:** Receita Federal do Brasil, dados abertos de CNPJ, competencia 2026-09, particao 1 de Estabelecimentos.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import polars as pl

from src import carregar, layout, perfilar, transformar, validar

DIR_RAW = RAIZ / "data" / "raw" / "2026-09"
DIR_CACHE = RAIZ / "data" / "cache"
pl.Config(fmt_str_lengths=60, tbl_rows=20)

## 1. Entrada

Os CSV da RFB vem sem cabecalho e em ISO-8859-1. A funcao `carregar` transcodifica uma unica vez para UTF-8 em cache e devolve um `LazyFrame` com todas as colunas como texto - a tipagem e etapa explicita depois, para nao mascarar os defeitos que queremos medir.

In [ ]:
lf_est = carregar.estabelecimentos(DIR_RAW / "Estabelecimentos1.zip", DIR_CACHE)
lf_cnae = carregar.cnaes(DIR_RAW / "Cnaes.zip", DIR_CACHE)
lf_mun = carregar.municipios(DIR_RAW / "Municipios.zip", DIR_CACHE)

total_particao = lf_est.select(pl.len()).collect().item()
print(f"estabelecimentos na particao 1: {total_particao:,}")
print(f"atributos: {len(layout.COLUNAS_ESTABELECIMENTOS)}")

## 2. Recorte do projeto

"Tecnologia" nao e um codigo CNAE - atravessa quatro secoes da CNAE 2.0. O recorte usa as divisoes 62 (servicos de TI) e 63 (servicos de informacao), ambas da secao J, filtradas por prefixo porque uma mesma empresa pode estar em 6201, 6202 ou 6204 conforme a escolha do contador.

In [ ]:
df = transformar.recortar(lf_est).collect()
print(f"UF={layout.UF_ALVO}, CNAE {'+'.join(layout.CNAE_ALVO_PREFIXOS)}: {df.height:,} estabelecimentos")
print(f"reducao: {100 * (1 - df.height / total_particao):.3f}% da particao")
df.head(3)

## 3. Valores ausentes

Ausencia na RFB aparece como campo vazio ou com espacos - as duas formas contam aqui.

In [ ]:
perfilar.ausentes(df)

## 4. Duplicidades

O CNPJ de 14 digitos nao existe pronto no arquivo: vem quebrado em basico (8), ordem (4) e digito verificador (2). Montar a chave e pre-requisito para medir duplicidade.

In [ ]:
df = df.with_columns(validar.expr_cnpj())
print(perfilar.duplicidades(df, "cnpj"))
print(perfilar.duplicidades(df, "cnpj_basico"), "  <- filiais do mesmo grupo, esperado")

## 5. Integridade: digito verificador

Validacao de dominio que quase nunca se faz em base cadastral, e que aqui e barata: o modulo 11 confere os dois digitos finais contra os doze primeiros.

In [ ]:
dv = df.select(validar.expr_dv_valido().alias("dv_ok"))
print(f"CNPJ com DV invalido: {int((~dv['dv_ok']).sum()):,} de {df.height:,}")

## 6. Codificacao de municipio

Armadilha classica da base: o codigo de municipio **nao e IBGE**, e uma tabela interna da RFB. Cruzar com qualquer fonte externa exige o de-para de `Municipios.zip`.

In [ ]:
mun = lf_mun.collect()
print(mun.filter(pl.col("descricao") == "SALVADOR"))
print("codigo IBGE de Salvador (2927408) existe na tabela da RFB?",
      mun.filter(pl.col("codigo") == "2927408").height > 0)

## 7. Distribuicoes relevantes

In [ ]:
cnae_nome = lf_cnae.collect().rename({"codigo": "cnae_fiscal_principal", "descricao": "cnae_descricao"})
vis = (
    df.with_columns(pl.col("cnae_fiscal_principal").str.zfill(7))
    .join(cnae_nome.with_columns(pl.col("cnae_fiscal_principal").str.zfill(7)), on="cnae_fiscal_principal", how="left")
    .with_columns(
        pl.col("situacao_cadastral").replace_strict(layout.SITUACAO_CADASTRAL, default="?").alias("situacao"),
        pl.col("municipio").replace_strict(dict(zip(mun["codigo"], mun["descricao"])), default="?").alias("municipio_nome"),
    )
)
display(perfilar.distribuicao(vis, "situacao"))
display(perfilar.distribuicao(vis, "cnae_descricao"))
display(perfilar.distribuicao(vis, "municipio_nome"))

## 8. Intervalo temporal

A RFB grava data como texto AAAAMMDD e usa `00000000` para ausencia; o parse nao estrito devolve nulo nesses casos.

In [ ]:
print(perfilar.intervalo_temporal(df, "data_inicio_atividade"))
print(perfilar.intervalo_temporal(df, "data_situacao_cadastral"))

## 9. Estatistica do dominio: participacao de MEI

Numero decisivo do projeto. MEI nao e um valor de `porte` - o campo so distingue micro, pequeno porte e demais. A marcacao vem de `Simples.zip`, o que custa uma juncao a mais.

Se a participacao for alta, esta provado que **filtrar por CNAE nao resolve**: a lista precisa ser ranqueada, nao apenas filtrada.

In [ ]:
lf_sim = carregar.simples(DIR_RAW / "Simples.zip", DIR_CACHE)
mei = (
    df.lazy()
    .join(lf_sim.select("cnpj_basico", "opcao_mei", "data_exclusao_mei"), on="cnpj_basico", how="left")
    .with_columns(((pl.col("opcao_mei") == "S") & pl.col("data_exclusao_mei").is_null()).fill_null(False).alias("is_mei"))
    .collect()
)
n_mei = int(mei["is_mei"].sum())
print(f"MEI: {n_mei:,} de {mei.height:,} ({100 * n_mei / mei.height:.1f}%)")

---

A sintese tecnica das cinco perguntas do item 6 esta em [`docs/atividade-01-dados.md`](../docs/atividade-01-dados.md).